<a href="https://colab.research.google.com/github/nikitask14/uci-bank-marketing-study/blob/main/NN_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**uci-bank-marketing-study**

**Step 1: Load the Python Libraries**

In [133]:
import pandas as pd
import numpy as np

**Step 2: Load the dataset**

In [134]:
# df = pd.read_csv("bank-additional-full.csv")
df = pd.read_csv("bank-additional-full.csv", sep = ";")

After inspecting the raw file we have discovered that data in the following format:

"value";"value";"value"

This causes pandas to never find the comma seperator it has been looking for and treats an entire physical row as one field.

**So, we have to tell the pd.read_csv() that delimiter in this file is ; and not ,**

So we use:
df = pd.read_csv("bank-additional-full.csv", sep = ";") instead of

df = pd.read_csv("bank-additional-full.csv")


In [135]:
df.shape

(41188, 21)

In [136]:
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


#### **Step 3: UCI Bank Marketing Data Profiling**
We are using the **UCI Bank Marketing dataset**, specifically:


*bank-additional-full.csv*


The dataset contains:


- 41,188 observations
- 21 columns


These consist of:

- 20 input variables
- 1 binary target variable, ***y***

The target indicates whether the client subscribed to a **term deposit**: yes / no

---

#### **Understanding the information available in the dataset**

The input variables can be grouped conceptually according to what kind of information they represent.

##### **1. Client information**

Examples include:


- age
- job
- marital
- education
- default
- housing
- loan


These variables describe the client and would generally be known before the current marketing interaction takes place.

---

##### **2. Previous campaign information**


- pdays
- previous
- poutcome


These variables describe the client's history from previous marketing campaigns.

For some clients, previous campaign information exists; for others, there was no previous contact.

This information would also be available before the current campaign interaction begins.

---

##### **3. Wider economic information**


- emp.var.rate
- cons.price.idx
- cons.conf.idx
- euribor3m
- nr.employed


These variables describe the wider economic environment rather than the individual client.

They may be available before the current interaction, depending on the exact prediction point.

---

##### **4. Current campaign/contact information**

Examples include:


- contact
- month
- day_of_week
- campaign
- duration


These variables are different because their availability depends on when the prediction is being made.

Some may be known once the current campaign is underway, while others, such as **duration**, can only be known after the contact itself has taken place.

This led to the key research-design insight:

> **The validity of a feature depends not only on whether it exists in the dataset, but also on whether it would actually be available at the moment the prediction is supposed to be made.**

---


 **Step 4: Research Problem Formulation**

 Research problem
Instead of treating the dataset simply as:

- **features → predict subscription to term deposit**

the study will compare two different information states.

---
**Problem A — Pre-campaign / pre-contact prediction**

Use only information that would be available before the current campaign interaction:

```
client information
+
previous campaign history
+
wider economic information
```
**Research question:**


How well can we predict whether a client will subscribe to a term deposit using only information available before the current campaign interaction?

---

**Problem B — Interaction-informed prediction**
Use the same pre-campaign information and add information generated during the current campaign/contact.

**Research question:**

How much does predictive performance change when current-campaign information is added?

**Main research question**

```Does adding current-campaign information improve our ability to predict the final term-deposit subscription outcome compared with using only information available before the current campaign interaction?```

This creates a controlled comparison between:

- pre-campaign information

vs
- pre-campaign + current-campaign information

The aim is to measure whether current-campaign information provides additional predictive signal.

**Why this project adds value**

This project is not simply another binary-classification exercise.

It introduces a new layer of research judgment by requiring us to reason about:

- what real-world decision the model is supposed to support,
- when the prediction is made,
- which information is legitimately available at that moment,
- how to design a fair controlled comparison,
- and what conclusions the experiment can and cannot support.
- At the same time, it extends the technical workflow by moving from a single-model experiment toward a controlled, repeatable comparison between two experimental conditions.


**Step 5: Inspecting Data types**

In [137]:
# info only tells us that pandas didn't recognise any entry as NaN
# but it could use ?, unknown or any other numerical identifier for
# missingness
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [138]:
df["age"].unique()

array([56, 57, 37, 40, 45, 59, 41, 24, 25, 29, 35, 54, 46, 50, 39, 30, 55,
       49, 34, 52, 58, 32, 38, 44, 42, 60, 53, 47, 51, 48, 33, 31, 43, 36,
       28, 27, 26, 22, 23, 20, 21, 61, 19, 18, 70, 66, 76, 67, 73, 88, 95,
       77, 68, 75, 63, 80, 62, 65, 72, 82, 64, 71, 69, 78, 85, 79, 83, 81,
       74, 17, 87, 91, 86, 98, 94, 84, 92, 89])

In [139]:
for column in df.columns:
  print(column, df[column].unique())

age [56 57 37 40 45 59 41 24 25 29 35 54 46 50 39 30 55 49 34 52 58 32 38 44
 42 60 53 47 51 48 33 31 43 36 28 27 26 22 23 20 21 61 19 18 70 66 76 67
 73 88 95 77 68 75 63 80 62 65 72 82 64 71 69 78 85 79 83 81 74 17 87 91
 86 98 94 84 92 89]
job ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']
marital ['married' 'single' 'divorced' 'unknown']
education ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']
default ['no' 'unknown' 'yes']
housing ['no' 'yes' 'unknown']
loan ['no' 'yes' 'unknown']
contact ['telephone' 'cellular']
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']
day_of_week ['mon' 'tue' 'wed' 'thu' 'fri']
duration [ 261  149  226 ... 1246 1556 1868]
campaign [ 1  2  3  4  5  6  7  8  9 10 11 12 13 19 18 23 14 22 25 16 17 15 20 56
 39 35 42 28 26 27 32 21 24 29 31 30 41 37 40 33 34 43]
pdays [999

We have identified:
-  "unknown" in job
- "unknown" in marital
- "unknown" in education
- "unknown" in default
- "unknown" in housing
- "unknown" in loan
- pdayds = 999 for this dataset represents no contact or unknonw.

In [140]:
#  counting unknowns and 999 in each columns where it is identified
print("job - ",(df["job"] == "unknown").sum())
print("marital -",(df["marital"] == "unknown").sum())
print("education -",(df["education"] == "unknown").sum())
print("default - ",(df["default"] == "unknown").sum())
print("housing - ",(df["housing"] == "unknown").sum())
print("loan - ",(df["loan"] == "unknown").sum())
print("pdays - ",(df["pdays"] == 999).sum())


job -  330
marital - 80
education - 1731
default -  8597
housing -  990
loan -  990
pdays -  39673


In [141]:
(df == "unknown").sum()

,0
age,0
job,330
marital,80
education,1731
default,8597
housing,990
loan,990
contact,0
month,0
day_of_week,0


Two useful distinctions:
- default is the one that really stands out as a substantial "unknown" category.
- education is noticeable, but much smaller.
- The rest are relatively limited.
- 999 is not missingness to “clean away”; it represents a meaningful condition: no prior contact.

**Step 6: Handling Numerical Unknown - 999**

In [142]:
#0 when pdays is 999, otherwise 1
df["previously_contacted"] = np.where(df["pdays"] == 999, 0, 1)

In [143]:
# 0 when 999, else retain the original column value
df["days_since_previous_contact"] = np.where(df["pdays"] == 999, 0, df["pdays"])

In [144]:
print(df["previously_contacted"].unique())
print(df["days_since_previous_contact"].unique())

[0 1]
[ 0  6  4  3  5  1 10  7  8  9 11  2 12 13 14 15 16 21 17 18 22 25 26 19
 27 20]


In [145]:
df.loc[df["pdays"] == 999,
       ["pdays", "previously_contacted",
        "days_since_previous_contact"]].head()

,pdays,previously_contacted,days_since_previous_contact
0,999,0,0
1,999,0,0
2,999,0,0
3,999,0,0
4,999,0,0


In [146]:
df = df.drop(columns = ["pdays"])

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          41188 non-null  int64  
 1   job                          41188 non-null  object 
 2   marital                      41188 non-null  object 
 3   education                    41188 non-null  object 
 4   default                      41188 non-null  object 
 5   housing                      41188 non-null  object 
 6   loan                         41188 non-null  object 
 7   contact                      41188 non-null  object 
 8   month                        41188 non-null  object 
 9   day_of_week                  41188 non-null  object 
 10  duration                     41188 non-null  int64  
 11  campaign                     41188 non-null  int64  
 12  previous                     41188 non-null  int64  
 13  poutcome        

**Step 7: Handling Categorical Unknown**

We retain unknown as a category in itself with removing it(we don't loss observations/examples) or using imputation as it would introduce incorrect information to the dataset.

Whether this has any predictive advantages remians to be seen and the model must learn from the data.

**Step 8: Target Encoding and Target Class Distribution**

In [148]:
# Checking unique values in the target
df["y"].unique()

array(['no', 'yes'], dtype=object)

In [149]:
df["y"] = np.where(df["y"] == "no", 0, 1)

In [150]:
# Understanding Unique Values in Target label y
df["y"].unique()

array([0, 1])

In [151]:
# Acual count of the number of 0 and 1 in the target
print((df["y"] == 0).sum())
print((df["y"]== 1).sum())

# 36548 + 4640



36548
4640


- Not Subscribed - 0 = 36548/41188 = 88.73%
- Subscribed - 1 = 4640/41188 = 11.26%

**Step 8: Dataset Splitting**

In [152]:
from sklearn.model_selection import train_test_split


In [153]:
#For the experimetal set of problem A, we delebirately remove the current campaign information
feature_column_A = [
    "age", "job", "marital",	"education", "default", "housing",	"loan",
    "previous"	,"poutcome"	,"previously_contacted",
    "days_since_previous_contact","emp.var.rate"	,"cons.price.idx"	,
    "cons.conf.idx",	"euribor3m"	,"nr.employed"	]
feature_column_B = [
      "age", "job", "marital",	"education", "default", "housing",	"loan",
      "contact", "month","day_of_week", "campaign", "duration",
    "previous"	,"poutcome"	,"previously_contacted",
    "days_since_previous_contact","emp.var.rate"	,"cons.price.idx"	,
    "cons.conf.idx",	"euribor3m"	,"nr.employed"	]



In [154]:
X_b = df[feature_column_B]
y = df["y"]
X_b

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,duration,previous,poutcome,previously_contacted,days_since_previous_contact,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,261,0,nonexistent,0,0,1.1,93.994,-36.4,4.857,5191.0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,149,0,nonexistent,0,0,1.1,93.994,-36.4,4.857,5191.0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,226,0,nonexistent,0,0,1.1,93.994,-36.4,4.857,5191.0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,151,0,nonexistent,0,0,1.1,93.994,-36.4,4.857,5191.0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,307,0,nonexistent,0,0,1.1,93.994,-36.4,4.857,5191.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,1,334,0,nonexistent,0,0,-1.1,94.767,-50.8,1.028,4963.6
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,1,383,0,nonexistent,0,0,-1.1,94.767,-50.8,1.028,4963.6
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,2,189,0,nonexistent,0,0,-1.1,94.767,-50.8,1.028,4963.6
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,1,442,0,nonexistent,0,0,-1.1,94.767,-50.8,1.028,4963.6


In [155]:
x_train_val, x_test_final, y_train_val, y_test_final = train_test_split(
    X_b,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y

)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_val,
    y_train_val,
    test_size = 0.20,
    random_state = 42,
    stratify = y_train_val

)

**Problem A** : **Execution**

In [156]:
# Now, create x_train_a, x_val_a, x_test_a as they should have 16 columns
# x_train, x_val and x_test_final are our master splits, containing all 21 predictor columns.
# Same rows, fewer columns.
x_train_A = x_train[feature_column_A]
x_val_A = x_val[feature_column_A]
x_test_A = x_test_final[feature_column_A]

TBRA: From the DataFrame x_train, select only the columns whose names are listed in feature_column_A, retain all the existing rows, and assign the resulting DataFrame to x_train_A.

In [157]:
print(x_train_A.shape)
print(x_val_A.shape)
print(x_test_A.shape)

(26360, 16)
(6590, 16)
(8238, 16)


In [158]:
# Separate the numerical and categorical predictor columns for Experiment A.
numerical_columns_A = ["age","previous", "previously_contacted",
                       "days_since_previous_contact","emp.var.rate"	,
                     "cons.price.idx"	, "cons.conf.idx",	"euribor3m"	,
                     "nr.employed"]
categorical_columns_A = ["job", "marital", "education", "default",
                       "housing",	"loan","poutcome"]

**Step 9: Preprocessing**

In [159]:
# from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (OneHotEncoder, StandardScaler)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [160]:
# Not doing any imputation and as we have already dealt with missingness
# We have also decided to use unknown as a cateogry of its own.
# So we don't use Simple Imputer Stregry like "mdeian or "most_frequent"
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown = "ignore"))
])

In [161]:
preprocessor_A = ColumnTransformer([
    ("numerical" , numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

In [162]:
from sklearn.linear_model import LogisticRegression
baseline_model = Pipeline([
    ("preprocessor", preprocessor_A),
    ("classification", LogisticRegression(max_iter = 1000))
])

baseline_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'previous',
                                                   'previously_contacted',
                                                   'days_since_previous_contact',
                                                   'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan',
                                                   'poutcome'])])),
                ('classification', LogisticRegression(max_iter=1000))])

In [163]:
baseline_model.fit(x_train_A, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'previous',
                                                   'previously_contacted',
                                                   'days_since_previous_contact',
                                                   'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan',
                                                   'poutcome'])])),
                ('classification', LogisticRegression(max_iter=1000))])

**Experiment A — Logistic Regression Baseline: Validation Set Evaluation**

In [164]:
prediction_val_A = baseline_model.predict(x_val_A)

In [165]:
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score,precision_score,recall_score)

In [166]:
def metrics(y_value, y_pred):
   cm = confusion_matrix(y_value, y_pred)
   accuracy = accuracy_score(y_value, y_pred)
   precision = precision_score(y_value, y_pred)
   recall = recall_score(y_value, y_pred)
   f1 = f1_score(y_value, y_pred)

   return cm, accuracy, precision, recall, f1

In [167]:
cm, accuracy, precision, recall, f1 = metrics(y_val, prediction_val_A)


print("Confusion Matrix:", cm)
print("Accuracy:", accuracy)
print("Precision", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix: [[5783   65]
 [ 602  140]]
Accuracy: 0.8987860394537177
Precision 0.6829268292682927
Recall: 0.18867924528301888
F1 Score: 0.29567053854276665


##### Interpretation - Experiment A (Logistic Regression)

- High accuracy (89.88%) is misleading due to class imbalance.
- Precision (68.29%) indicates reasonably reliable positive predictions.
- Low recall (18.87%) indicates that the model misses 602 out of 742 actual subscribers.
- Low F1-score (29.57%) reflects the poor balance between precision and recall.

Overall, the model identifies non-subscribers well but has limited ability to identify actual subscribers using pre-campaign information.

**Experiment B — Logistic Regression Baseline: Validation Set Evaluation**

In [171]:
numerical_columns_B = ["age","previous", "previously_contacted",
                       "days_since_previous_contact",
                        "campaign", "duration","emp.var.rate"	,
                     "cons.price.idx"	, "cons.conf.idx",	"euribor3m"	,
                     "nr.employed"]
categorical_columns_B = ["job", "marital", "education", "default",
                       "housing",	"loan","contact", "month","day_of_week",
                         "poutcome"]

In [ ]:
preprocessor_B = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns_B),
    ("categorical", categorical_pipeline, categorical_columns_B),

])

In [ ]:
baseline_model_B = Pipeline([
    "preprocessor"
])

In [168]:
baseline_model.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'previous',
                                                   'previously_contacted',
                                                   'days_since_previous_contact',
                                                   'emp.var.rate',
                                                   'cons.price.idx',
                                                   'cons.conf.idx', 'euribor3m',
                                                   'nr.employed']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan',
                                                   'poutcome'])])),
                ('classification', LogisticRegression(max_iter=1000))])

In [169]:
prediction_val_B = baseline_model.predict(x_val)

In [170]:
cm_B, accuracy_B, precision_B, recall_B, f1_B = metrics(y_val, prediction_val_B)
print("Confusion Matrix:", cm_B)
print("Accuracy:", accuracy_B)
print("Precision", precision_B)
print("Recall:", recall_B)
print("F1 Score:", f1_B)

Confusion Matrix: [[5783   65]
 [ 602  140]]
Accuracy: 0.8987860394537177
Precision 0.6829268292682927
Recall: 0.18867924528301888
F1 Score: 0.29567053854276665
